# DA6401 Assignment 3 — Transformer for Machine Translation


In [6]:
import shutil, os
for f in ['model.py', 'dataset.py', 'train.py']:
    shutil.copy(f'/kaggle/input/datasets/saiaravindsrivatsav/dataset2/{f}', f'./{f}')

In [ ]:
# Step 1: Install everything
!pip install -q wandb datasets sacrebleu gdown spacy
!python -m spacy download de_core_news_sm -q
!python -m spacy download en_core_web_sm -q

In [7]:
# Step 2: clone your fork of the skeleton (or upload files directly)
import os
# If you upload model.py / dataset.py / train.py to Kaggle as a dataset,
# copy them here. Otherwise clone your GitHub repo:
# !git clone https://github.com/YOUR_USERNAME/da6401_assignment_3.git
# %cd da6401_assignment_3

# Check files are present
print(os.listdir('.'))

['.virtual_documents', 'dataset.py', 'train.py', 'model.py']


In [8]:
# Step 3: log in to W&B
import wandb
wandb.login()   # paste your API key when prompted

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aravindsrivatsav234 (aravindsrivatsav234-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
# Step 4: verify GPU
import torch
print(torch.cuda.device_count(), 'GPU(s)')
print(torch.cuda.get_device_name(0))

0 GPU(s)


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Step 5: run the BASE model only (gets you the submission checkpoint)
# This takes ~1.5 hrs on T4 x2 for 30 epochs.
!python train.py --experiments base --epochs 30 --wandb_project da6401-a3

In [11]:
# Step 6: run the remaining 4 W&B experiments
# Each pair of runs ~45-60 min. Run one at a time to avoid session timeout.

!python train.py --experiments noam  --epochs 25 --wandb_project da6401-a3
!python train.py --experiments scale --epochs 20 --wandb_project da6401-a3
!python train.py --experiments pe    --epochs 25 --wandb_project da6401-a3
!python train.py --experiments smooth --epochs 25 --wandb_project da6401-a3

Device: cuda | GPUs: 2
Loading data...
src vocab: 7853 | trg vocab: 5893

=== EXP 1: Noam Scheduler ===
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: aravindsrivatsav234 (aravindsrivatsav234-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260511_090517-hidtqdqg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run noam_scheduler
wandb: ⭐️ View project at https://wandb.ai/aravindsrivatsav234-iit-madras/da6401-a3
wandb: 🚀 View run at https://wandb.ai/aravindsrivatsav234-iit-madras/da6401-a3/runs/hidtqdqg
[noam_scheduler] params: 8,981,253
Epoch   1 | train=8.1806 val=7.4043
Epoch   2 | train=6.6319 val=5.6767
Epoch   3 | train=5.3255 val=4.8689
Epoch   4 | train=4.7311 val=4.3795
That's 100 

In [10]:
# Step 7: quick sanity check before uploading weights
import torch
from model import Transformer

# temporarily point to local file to test the infer() path
ckpt = torch.load('best_model.pt', map_location='cpu')
model = Transformer(src_vocab=ckpt['src_vocab'], trg_vocab=ckpt['trg_vocab'])
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

test_sentence = 'Ein Mann sitzt auf einem Stuhl.'
print('DE:', test_sentence)
print('EN:', model.infer(test_sentence))

DE: Ein Mann sitzt auf einem Stuhl.
EN: a man is sitting in a chair .


In [ ]:
# ── Step 8: upload best_model.pt to Google Drive, then paste the file ID ─────
# 1. Download best_model.pt from Kaggle output panel (right sidebar)
# 2. Upload it to your Google Drive
# 3. Right-click → 'Get link' → copy the FILE ID (the part after /d/ in the URL)
# 4. Open model.py and replace PASTE_YOUR_GOOGLE_DRIVE_FILE_ID_HERE with that ID
# 5. Make sure the Drive file is shared as 'Anyone with the link can view'
print('See INSTRUCTIONS.md Step 4 for details')

In [9]:
# Step 9: final autograder simulation 
# This is exactly what the autograder runs. Must work before submitting.
import torch
from model import Transformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Transformer().to(device)   # no args — downloads weights from Drive
model.eval()

german_sentence = 'Zwei Hunde spielen im Schnee.'
english_sentence = model.infer(german_sentence)
print('Input :', german_sentence)
print('Output:', english_sentence)

Input : Zwei Hunde spielen im Schnee.
Output: two dogs play in the snow .


In [5]:
%%writefile train.py

import argparse
import os
import random

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import wandb
from tqdm import tqdm

matplotlib.use("Agg")

from dataset import load_data
from model import (
    EOS_IDX,
    PAD_IDX,
    SOS_IDX,
    LabelSmoothingLoss,
    NoamScheduler,
    Transformer,
)

# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def save_checkpoint(model, path, src_vocab, trg_vocab, cfg):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "src_vocab": src_vocab,
            "trg_vocab": trg_vocab,
            "config": cfg,
        },
        path,
    )


# ─────────────────────────────────────────────
# Train / Eval
# ─────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, criterion, device, log_grads=False, step_offset=0):
    model.train()
    total_loss = 0
    step = step_offset
    grad_log = []

    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        trg_in = trg[:, :-1]
        trg_out = trg[:, 1:]

        logits = model(src, trg_in)
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B * T, V), trg_out.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        if log_grads and step < 1000:
            # gradient norms of Q and K weights across encoder layers
            q_norm, k_norm, count = 0.0, 0.0, 0
            inner_model = model.module if isinstance(model, torch.nn.DataParallel) else model
            for layer in inner_model.encoder.layers:
                if layer.self_attn.w_q.weight.grad is not None:
                    q_norm += layer.self_attn.w_q.weight.grad.norm().item() ** 2
                    k_norm += layer.self_attn.w_k.weight.grad.norm().item() ** 2
                    count += 1
            if count:
                grad_log.append({"step": step, "q_grad_norm": q_norm ** 0.5, "k_grad_norm": k_norm ** 0.5})

        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        step += 1

    return total_loss / len(loader), step, grad_log


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        trg_in = trg[:, :-1]
        trg_out = trg[:, 1:]
        logits = model(src, trg_in)
        B, T, V = logits.shape
        total_loss += criterion(logits.reshape(B * T, V), trg_out.reshape(-1)).item()
    return total_loss / len(loader)


@torch.no_grad()
def compute_bleu(model, records, n=400):
    """Compute sacrebleu on up to n samples from records list."""
    import sacrebleu
    hyps, refs = [], []
    model.eval()
    for r in records[:n]:
        hyps.append(model.infer(r["de"]))
        refs.append(r["en"])  # raw text — sacrebleu handles tokenization internally
    return sacrebleu.corpus_bleu(hyps, [refs], tokenize="13a").score


@torch.no_grad()
def avg_confidence(model, loader, device, n_batches=20):
    """Average softmax prob of the correct token (label smoothing experiment)."""
    model.eval()
    total, count = 0.0, 0
    for i, (src, trg) in enumerate(loader):
        if i >= n_batches:
            break
        src, trg = src.to(device), trg.to(device)
        trg_in = trg[:, :-1]
        trg_out = trg[:, 1:]
        logits = model(src, trg_in)
        probs = torch.softmax(logits, dim=-1)
        B, T, V = probs.shape
        flat_probs = probs.reshape(B * T, V)
        flat_target = trg_out.reshape(-1)
        mask = flat_target != PAD_IDX
        correct_probs = flat_probs[mask].gather(1, flat_target[mask].unsqueeze(1)).squeeze(1)
        total += correct_probs.sum().item()
        count += correct_probs.numel()
    return total / count if count else 0.0


# ─────────────────────────────────────────────
# Attention visualisation (experiment 2.3)
# ─────────────────────────────────────────────

@torch.no_grad()
def visualize_attention(model, sentence_de, trg_vocab, device):
    model.eval()
    from model import UNK_IDX

    tokens_de = [t.text.lower() for t in model.spacy_de.tokenizer(sentence_de)]
    src_ids = [SOS_IDX] + [model.src_vocab.get(t, UNK_IDX) for t in tokens_de] + [EOS_IDX]
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    src_mask = model.make_src_mask(src)
    model.encoder(src, src_mask)

    last_layer = model.encoder.layers[-1]
    attn = last_layer.self_attn.attn_weights[0].cpu().numpy()  # (heads, T, T)
    labels = ["<sos>"] + tokens_de + ["<eos>"]

    num_heads = attn.shape[0]
    ncols = 4
    nrows = (num_heads + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = axes.flatten()

    for h in range(num_heads):
        ax = axes[h]
        im = ax.imshow(attn[h], cmap="viridis", aspect="auto")
        ax.set_title(f"Head {h + 1}", fontsize=9)
        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=6)
        ax.set_yticklabels(labels, fontsize=6)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for h in range(num_heads, len(axes)):
        axes[h].axis("off")

    plt.suptitle("Encoder Last Layer — Per-Head Attention", fontsize=11)
    plt.tight_layout()
    path = "attention_heads.png"
    plt.savefig(path, dpi=120)
    plt.close()
    return path


# ─────────────────────────────────────────────
# Core training runner
# ─────────────────────────────────────────────

def run(
    run_name,
    train_loader,
    val_loader,
    test_records,
    src_vocab,
    trg_vocab,
    device,
    epochs=25,
    warmup_steps=3000,
    use_noam=True,
    fixed_lr=1e-4,
    smoothing=0.1,
    pe_type="sinusoidal",
    use_scale=True,
    log_grads=False,
    wandb_project="da6401-a3",
):
    cfg = dict(
        d_model=256, num_heads=8, num_layers=3, d_ff=512,
        max_len=150, dropout=0.1, pe_type=pe_type, use_scale=use_scale,
    )

    wandb.init(project=wandb_project, name=run_name, config={**cfg, "smoothing": smoothing, "noam": use_noam})

    model = Transformer(src_vocab=src_vocab, trg_vocab=trg_vocab, **cfg).to(device)

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    inner = model.module if isinstance(model, nn.DataParallel) else model
    print(f"[{run_name}] params: {count_params(inner):,}")

    criterion = LabelSmoothingLoss(len(trg_vocab), smoothing=smoothing)
    optimizer = torch.optim.Adam(model.parameters(), lr=0 if use_noam else fixed_lr, betas=(0.9, 0.98), eps=1e-9)

    scheduler = None
    if use_noam:
        scheduler = NoamScheduler(optimizer, d_model=cfg["d_model"], warmup_steps=warmup_steps)

    best_val_loss = float("inf")
    best_bleu = 0.0
    ckpt_path = f"best_{run_name}.pt"
    global_step = 0

    for epoch in range(1, epochs + 1):
        train_loss, global_step, grad_log = train_epoch(
            model, train_loader, optimizer, scheduler, criterion, device,
            log_grads=log_grads, step_offset=global_step
        )
        val_loss = eval_epoch(model, val_loader, criterion, device)

        log = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss}
        if scheduler:
            log["lr"] = scheduler.get_lr()
        else:
            log["lr"] = fixed_lr

        for g in grad_log:
            wandb.log({"step": g["step"], "q_grad_norm": g["q_grad_norm"], "k_grad_norm": g["k_grad_norm"]})

        if epoch % 5 == 0 or epoch == epochs:
            bleu = compute_bleu(inner, test_records, n=300)
            log["val_bleu"] = bleu
            conf = avg_confidence(model, val_loader, device)
            log["pred_confidence"] = conf
            print(f"Epoch {epoch:3d} | train={train_loss:.4f} val={val_loss:.4f} BLEU={bleu:.2f} conf={conf:.4f}")
            if bleu > best_bleu:
                best_bleu = bleu
                save_checkpoint(inner, ckpt_path, src_vocab, trg_vocab, cfg)
        else:
            print(f"Epoch {epoch:3d} | train={train_loss:.4f} val={val_loss:.4f}")

        wandb.log(log)

    wandb.log({"best_bleu": best_bleu})
    wandb.finish()
    return ckpt_path, best_bleu


# ─────────────────────────────────────────────
# Main — runs all experiments sequentially
# ─────────────────────────────────────────────

def main(args):
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")

    print("Loading data...")
    train_loader, val_loader, test_loader, src_vocab, trg_vocab, spacy_de, spacy_en, test_records = load_data(batch_size=256)
    print(f"src vocab: {len(src_vocab)} | trg vocab: {len(trg_vocab)}")

    common = dict(
        train_loader=train_loader, val_loader=val_loader, test_records=test_records,
        src_vocab=src_vocab, trg_vocab=trg_vocab, device=device,
        wandb_project=args.wandb_project,
    )

    # ── Experiment 1: Noam vs Fixed LR (Section 2.1) ──────────────────────────
    if "all" in args.experiments or "noam" in args.experiments:
        print("\n=== EXP 1: Noam Scheduler ===")
        run("noam_scheduler", use_noam=True, epochs=25, **common)
        run("fixed_lr_1e4", use_noam=False, fixed_lr=1e-4, epochs=25, **common)

    # ── Experiment 2: Scaling factor ablation (Section 2.2) ───────────────────
    if "all" in args.experiments or "scale" in args.experiments:
        print("\n=== EXP 2: Scaling Factor Ablation ===")
        run("with_scale", use_scale=True, epochs=20, log_grads=True, **common)
        run("no_scale", use_scale=False, epochs=20, log_grads=True, **common)

    # ── Base model + attention visualisation (Section 2.3) ────────────────────
    if "all" in args.experiments or "base" in args.experiments:
        print("\n=== EXP BASE: Full Training + Attention Rollout ===")
        best_path, best_bleu = run("base_model", use_noam=True, epochs=args.epochs, **common)
        print(f"Best BLEU: {best_bleu:.2f}  saved → {best_path}")

        # visualise attention from best checkpoint
        ckpt = torch.load(best_path, map_location="cpu")
        vis_model = Transformer(src_vocab=src_vocab, trg_vocab=trg_vocab,
                                d_model=256, num_heads=8, num_layers=3, d_ff=512).to(device)
        vis_model.load_state_dict(ckpt["model_state_dict"])
        sample_de = test_records[0]["de"]
        attn_img = visualize_attention(vis_model, sample_de, trg_vocab, device)

        wandb.init(project=args.wandb_project, name="attention_rollout")
        wandb.log({"attention_heads": wandb.Image(attn_img, caption=sample_de)})
        wandb.finish()

        # rename best checkpoint to the final submission name
        import shutil
        shutil.copy(best_path, "best_model.pt")
        print("Saved best_model.pt — upload this to Google Drive.")

    # ── Experiment 4: Sinusoidal vs Learned PE (Section 2.4) ─────────────────
    if "all" in args.experiments or "pe" in args.experiments:
        print("\n=== EXP 4: Positional Encoding ===")
        run("sinusoidal_pe", pe_type="sinusoidal", epochs=25, **common)
        run("learned_pe", pe_type="learned", epochs=25, **common)

    # ── Experiment 5: Label Smoothing (Section 2.5) ───────────────────────────
    if "all" in args.experiments or "smooth" in args.experiments:
        print("\n=== EXP 5: Label Smoothing ===")
        run("smooth_0.1", smoothing=0.1, epochs=25, **common)
        run("smooth_0.0", smoothing=0.0, epochs=25, **common)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--experiments", nargs="+", default=["all"],
                        choices=["all", "base", "noam", "scale", "pe", "smooth"])
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--wandb_project", type=str, default="da6401-a3")
    args = parser.parse_args()
    main(args)

Overwriting train.py


In [4]:
!python train.py --experiments scale --epochs 20 --wandb_project da6401-a3

Traceback (most recent call last):
  File "/kaggle/working/train.py", line 16, in <module>
    from dataset2 import load_data
ModuleNotFoundError: No module named 'dataset2'
